In [1]:
import json
import re
from pathlib import Path
from typing import List, Tuple, Optional
import pandas as pd
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
fs = sorted(Path("../data/json2").glob("*.json"))
print(f"{len(fs)} threads")

10257 threads


In [3]:
from IPython.display import display
# load output from
df3 = pd.read_parquet("../outputs/df_links4.parquet")
df3

,score,n_links,n_comments,comments,thread_urls,first_link_utc,last_link_utc,url
title,,,,,,,,
Worth the Candle,1453,110,110,[{'author_flair_text': 'Self-Appointed Court S...,[https://reddit.com/r/rational/comments/7228po...,2017-07-28 13:17:36,2024-04-17 19:03:01,[https://archiveofourown.org/works/11478249/ch...
"Alexander Wales - The Metropolitan Man, Shadows of the Limelight",1408,22,22,[{'author_flair_text': 'Time flies like an arr...,[https://reddit.com/r/rational/comments/galt0r...,2015-04-18 18:13:47,2021-04-29 20:00:04,[https://www.patreon.com/alexanderwales]
Mother of Learning,1063,101,101,"[{'author_flair_text': 'Utopian Smut Peddler',...",[https://reddit.com/r/rational/comments/k8i7jg...,2014-07-26 08:21:12,2024-05-21 07:04:42,[https://www.fictionpress.com/s/2961893/1/Moth...
A Practical Guide to Evil,1025,92,92,[{'author_flair_text': 'Ankh-Morpork City Watc...,[https://reddit.com/r/rational/comments/ako900...,2016-04-05 18:53:21,2024-08-09 10:10:21,"[https://practicalguidetoevil.wordpress.com/, ..."
Worm,925,83,83,[{'author_flair_text': 'Emergency Mustelid Hol...,[https://reddit.com/r/rational/comments/k8i7jg...,2014-01-27 02:09:42,2024-11-22 21:39:11,"[https://parahumans.wordpress.com/, https://pa..."
...,...,...,...,...,...,...,...,...
The Incredibles,-6,1,1,"[{'author_flair_text': None, 'body': 'I've act...",[https://reddit.com/r/rational/comments/7cneg5...,2017-11-13 20:58:01,2017-11-13 20:58:01,[http://www.imdb.com/title/tt0317705/?ref_=fn_...
"Steelheart (The Reckoners): 9780385743570: Sanderson, Brandon: Books",-6,1,1,"[{'author_flair_text': None, 'body': 'I've act...",[https://reddit.com/r/rational/comments/7cneg5...,2017-11-13 20:58:01,2017-11-13 20:58:01,[https://www.amazon.com/Steelheart-Reckoners-B...
Orthogonality thesis,-8,2,2,"[{'author_flair_text': None, 'body': 'I don't ...",[https://reddit.com/r/rational/comments/3vc0si...,2015-12-04 18:17:07,2016-07-11 16:27:30,[https://wiki.lesswrong.com/wiki/Orthogonality...


## Extra get a llm summary of each link [WIP]

Grab all md's that mention a story, ask claude to summarize

We could also get total karma per mention

In [4]:
import dotenv
import os

dotenv.load_dotenv()
from openai import OpenAI
from openai import AsyncOpenAI
client = AsyncOpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ['OPENROUTER_API_KEY'],
)

import tiktoken
# https://openrouter.ai/models?fmt=cards&order=newest&supported_parameters=structured_outputs
MODEL_NAME = "gpt-4o-mini" # ?b
cost = 0.150 / 1e6
# MODEL_NAME = "google/gemini-flash-1.5" # json fail... should work though
MODEL_NAME = "google/gemini-flash-1.5" 
# MODEL_NAME = "qwen/qwen-2.5-72b-instruct"
# MODEL_NAME = "cohere/command-r" # json fail
# MODEL_NAME = "meta-llama/llama-3.3-70b-instruct" #json fail
enc = tiktoken.encoding_for_model('gpt-4')

In [5]:
# load all md posts 
md_posts = []
fs = sorted(Path("../data/cache2").glob("*.md"))
for f in tqdm(fs):
    s = f.open().read()
    md_posts.append(s)


# order by date
def md2date(s: str) -> str:
    return s.split('* Created: ')[1].split('\n')[0]

md_posts = sorted(md_posts, key=md2date)
md2date(md_posts[0]), md2date(md_posts[-1])

  0%|          | 0/10263 [00:00<?, ?it/s]

('2009-11-25T02:34:03', '2024-12-23T15:00:14')

In [6]:

def get_post_context(urls: List[str], char_budget=100000, min_size=1000) -> str:
    assert len(urls) > 0

    matches = []
    for ii, post in enumerate(md_posts):
        for url in urls:
            if url in post:
                matches.append(post)
                break

    budget_pp = char_budget / len(matches)
    budget_pp = max(budget_pp, min_size)
    s = ""
    for i in range(len(matches)):
        post = matches[i]

        for url in urls:
            if url in post:
                ind = post.index(url)

        i0 = int(max(0, ind - budget_pp // 4))
        i1 = int(min(len(post), ind + budget_pp // 4 * 3))
        post_chunk = post[i0:i1]
        if i0 > 0:
            post_chunk = "..." + post_chunk
        if i1 < len(post):
            post_chunk = post_chunk + "..."

        s += f"\n\n----- Thread {ii} -----\n\n" + post_chunk

    # if too large get first N//2 and last N//2
    if len(s) > char_budget:
        s = s[:char_budget // 2] + "..." + s[-char_budget // 2:]
    return s


In [7]:


from pydantic import BaseModel, Field


class FictionInfo(BaseModel):
    title: str = Field(description="Title of the fiction")
    description: str = Field(description="A few paragraphs of very concise, informative, dense, description of the fiction")
    tags: List[str] = Field(
        description="""Long list of common descriptors: format (web serial, fanfic, lightnovel, short, complete, comic), genre (scifi, fantasy)
        Key elements (rational, timeloop, litrpg, progression, cultivation, isekai)
        Content notes (grimdark, romance, harem, queer, funny, NSFW)
        """
    )

    # status: str = Field(description="complete/ongoing/hiatus/abandoned")
    # type: str = Field(description='e.g. fanfiction, original, comic, etc.')

    reviews_quotes: List[str] = Field(
        description="Directly and fully quote excerpts from every single readers' comments about the fiction"
    )
    reviews_summary: str = Field(
        description="Structured, dry, and concise summary of reviews"
    )
    reccomendations: str = Field(
        description="Why readers recommend the fiction"
    )
    disrecommendations: str = Field(
        description="Why readers disrecommend the fiction"
    )
    why: str = Field(
        description="Why/when might readers of r/rational like the fiction"
    )
    if_you_liked_x_you_will_like_this: List[str] = Field(
        description="Fans of X will also like the fiction. List all examples that readers mention wrt to the fiction."
    )

    rating_quality: float = Field(
        # ge=0.0, le=10.0,
        description="Overall user sentiment of readers towards the fiction (out of 10), it's important to be consistent and use the same scale for all fictions"
    )
    rating_rationality: float = Field(
        # ge=0.0, le=10.0,
        description="Systematic worldbuilding, character competence, logical consistency. Where HPMOR is a 10 and Worm is a 5."
    )
    rating_writing: float
    rating_plot: float
    rating_character: float
    rating_worldbuilding: float

    # TODO I want to know, how long is it, is it complete, is it a fiction, article, comic, etc.

In [8]:
from openai.lib._pydantic import to_strict_json_schema

schema = to_strict_json_schema(FictionInfo)
schema = json.dumps(schema)
# print(schema)

In [9]:

async def get_llm_summary(title: str, urls: str, context: str):
    chat_completion = await client.beta.chat.completions.parse(
        messages=[
            {
                "role": "system",
                "content": f"You are Gwern Branwern, an internet librarian who specializes in rational fiction. You are summarising community recommendations from r/rational into a dry, informative, concise, and structured format for your own personal notes. Because it's private you can be consise, frank, and opinionated. Ignoring any authors promotion. You  answer in JSON. Here's the json schema you must adhere to:\n<schema>\n{schema}\n",
            },
            {
                "role": "user",
                "content": f"""Using the given structure, summarize the parts of the discussion which talk about the fiction: {title} (urls: {urls}).

### Discussion:

{context}

### Instructions

You are Gwern Branwern, using the given structure, summarize the above discussion of the fiction {title} (urls: {urls}).""",
            },
        ],
        model=MODEL_NAME,
        response_format=FictionInfo,
    )

    tokens = chat_completion.usage.prompt_tokens
    return tokens, chat_completion.choices[0].message.parsed.__dict__

In [10]:
model_name_ds = re.sub(r'[^\w\s]', '', MODEL_NAME)

output_dir = Path(f'../outputs/llm2/{model_name_ds}')
output_dir.mkdir(exist_ok=True, parents=True)
output_dir

PosixPath('../outputs/llm2/googlegeminiflash15')

In [11]:
# import shutil
# shutil.rmtree(output_dir, ignore_errors=True)

from pydantic import ValidationError

In [12]:
import asyncio
import json
import os


In [18]:
import asyncio

async def process_item(title, urls, f):
    if f.exists():
        return json.load(f.open())
    context = get_post_context(urls, char_budget=120000)
    try:
        f_tokens, llm_data = await get_llm_summary(title, urls, context)
    except ValidationError as e:
        print(e)
        return None
    except ValueError as e:
        print(e)
        return None
    

    llm_data['context'] = context
    llm_data['title2'] = title
    llm_data['model'] = MODEL_NAME
    
    # print(
    #     f"Input Tokens: {f_tokens}. Input Cost: {cost * f_tokens:.6f} USD, for title=`{title}`"
    # )
    with f.open("w") as fo:
        json.dump(llm_data, fo)


    return llm_data

In [22]:
# async def main(df3):
#     tasks = []

#     for i in range(len(df3)):
#         title = df3.index[i]
#         urls = df3.iloc[i].url

#         fs_title = re.sub(r'[^\w\s]', '', title)[:200]
        
#         f = output_dir / f"{i}_{fs_title}.json"
#         tasks.append(process_item(title, urls, f))

#     llm_info = await asyncio.gather(*tasks)
#     print(llm_info)
#     return llm_info

async def main(df3, batch_size=300):
    all_llm_info = []
    total_tasks = len(df3)

    for start in range(0, total_tasks, batch_size):
        end = min(start + batch_size, total_tasks)
        batch = df3.iloc[start:end]
        
        tasks = []
        for i, (title, row) in enumerate(batch.iterrows(), start=start):
            urls = row.url
            fs_title = re.sub(r'[^\w\s]', '', title)[:200]
            f = output_dir / f"{i}_{fs_title}.json"
            tasks.append(process_item(title, urls, f))

        batch_llm_info = await asyncio.gather(*tasks)
        all_llm_info.extend(batch_llm_info)
        
        print(f"Processed batch {start//batch_size + 1} of {(total_tasks - 1)//batch_size + 1}")

    print("All batches processed")
    return all_llm_info

# Run the async main function
df4 = df3#.sample(200, random_state=42)
loop = asyncio.get_event_loop()
task = loop.create_task(main(df4))
value = task.result()
value

InvalidStateError: Result is not set.

1 validation error for FictionInfo
  Invalid JSON: EOF while parsing a string at line 1 column 6739 [type=json_invalid, input_value='{"description": "Multipl...ie gold mages have been', input_type=str]
    For further information visit https://errors.pydantic.dev/2.10/v/json_invalid
Unable to parse chat completion with error: {'message': 'The operation was aborted', 'code': 500}
Unable to parse chat completion with error: {'message': 'The operation was aborted', 'code': 500}
Unable to parse chat completion with error: {'message': 'The operation was aborted', 'code': 500}
Unable to parse chat completion with error: {'message': 'The operation was aborted', 'code': 500}
Unable to parse chat completion with error: {'message': 'The operation was aborted', 'code': 500}
1 validation error for FictionInfo
  Invalid JSON: EOF while parsing a string at line 1 column 8114 [type=json_invalid, input_value='{"description": "A Moder...", "  ,  ", "  ,  "}```', input_type=str]
    For further informat

In [34]:
llm_info = task.result()
llm_info = [x for x in llm_info if x is not None]
df_llm = pd.DataFrame(llm_info)
df_llm


,title,description,tags,reviews_quotes,reviews_summary,reccomendations,disrecommendations,why,if_you_liked_x_you_will_like_this,rating_quality,rating_rationality,rating_writing,rating_plot,rating_character,rating_worldbuilding,context,title2,model
0,Worth the Candle,"Worth the Candle is a long, original, complete...","[web serial, LitRPG, isekai, portal fantasy, o...","[This is a self-insert litRPG portal fantasy, ...","Overall, Worth the Candle received overwhelmin...",Readers frequently praised the story's compell...,Some readers found the sheer length of the wor...,Readers of r/rational would likely enjoy Worth...,"[Mother of Learning, HPMOR, Worm, Erfworld, Lu...",9.0,7.0,8.5,8.5,8.0,9.0,\n\n----- Thread 10262 -----\n\n## [RT][WIP] W...,Worth the Candle,google/gemini-flash-1.5
1,"Alexander Wales - The Metropolitan Man, Shadow...","Alexander Wales's works, including *Shadows of...","[web serial, rational fiction, fantasy, progre...",[Damn. That was like reading some of Sanderson...,"The reviews are overwhelmingly positive, with ...",Readers strongly recommend Alexander Wales's w...,Some readers express concerns about the infreq...,Readers of r/rational would likely enjoy Alexa...,"[Brandon Sanderson, Harry Potter, Worm, Mother...",9.0,7.5,9.0,9.0,8.5,8.0,\n\n----- Thread 10262 -----\n\n## [HF][DC] Sh...,"Alexander Wales - The Metropolitan Man, Shadow...",google/gemini-flash-1.5
2,Mother of Learning,Mother of Learning is a web serial about Zoria...,"[web serial, original fiction, fantasy, time l...","[With dread but cautious optimism, Enthusiasti...",Generally very positive; praised for clever pr...,"Readers praise its consistent internal logic, ...",Some readers find the writing style to be roug...,Readers of r/rational will appreciate the stor...,"[Time Braid, HPMOR, Worm]",9.0,7.5,7.0,9.0,8.5,9.0,\n\n----- Thread 10262 -----\n\n## [HF] Mother...,Mother of Learning,google/gemini-flash-1.5
3,A Practical Guide to Evil,A Practical Guide to Evil is a long-running we...,"[web serial, fantasy, complete, grimdark, roma...",[It's probably one of the better written ones ...,"Generally very positive, praising the writing,...","Readers praise its well-developed characters, ...",Some readers find the frequent cliffhangers ir...,Readers of r/rational will likely enjoy this f...,"[Worm, Shadows of the Limelight]",9.5,7.0,8.0,9.0,9.0,9.0,\n\n----- Thread 10262 -----\n\n## [RT] A Prac...,A Practical Guide to Evil,google/gemini-flash-1.5
4,Worm,"Worm is a long, complete web serial (1.7 milli...","[web serial, complete, superhero, supervillain...",[I wouldn't really describe Worm as a rational...,"Worm receives overwhelmingly positive reviews,...",Readers recommend Worm for its compelling char...,"Some find the story too grim, dark, and depres...",Readers of r/rational might enjoy Worm for its...,"[HPMOR, A Practical Guide to Evil, Other Wildb...",9.0,7.0,8.5,8.5,8.0,9.5,\n\n----- Thread 10262 -----\n\n## Worm\n\n* A...,Worm,google/gemini-flash-1.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15027,The Incredibles,The Incredibles is mentioned in passing in a d...,"[web serial, scifi, superhero, deconstruction,...",[> Why is it called *Worm* instead of *Parahum...,While initially compared to The Incredibles fo...,The mention of The Incredibles in the context ...,Some users find the initial comparison to The ...,Readers interested in exploring the various wa...,[Worm],7.0,7.0,7.0,7.0,6.0,7.0,\n\n----- Thread 10262 -----\n\n## Worm Reread...,The Incredibles,google/gemini-flash-1.5
15028,Steelheart (The Reckoners),Steelheart is a superhero deconstruction novel...,"[novel, complete, superhero, supervillain, dec...",[It sure looks derivative. I've seen these tro...,Readers' opinions on Steelheart were divided. ...,Some readers appreciated Steelheart's subvers...,"Some readers found Steelheart derivative, citi...",Readers of r/rational might enjoy Steelheart i...,"[Worm, The Incredibles, Watchmen, Megamind, Mi...",7.

In [25]:
# also join with df4
df_llm2 = (df_llm
          #  .drop(columns=['url'])
           .set_index('title2')
           .merge(df3, left_index=True, right_index=True)
              .sort_values('score', ascending=False)
            #   .drop(columns=['comments'])
            .reset_index(names='title2')
)
# df_llm2
# then display as table
# TODO check correlation between score and ratings
df_llm2_num = df_llm2.select_dtypes(include=['float64', 'int64'])
df_llm2_num['score_mean'] = df_llm2_num['score'] / df_llm2_num['n_links']
cc = df_llm2_num.iloc[:150]
c = cc.corr()['score']
print(f"Correlation with score ↑:\n{c} [n={len(cc)}] model={MODEL_NAME}")

Correlation with score ↑:
rating_quality          0.207542
rating_rationality      0.046665
rating_writing          0.113297
rating_plot             0.213538
rating_character        0.162677
rating_worldbuilding    0.142387
score                   1.000000
n_links                 0.813041
n_comments              0.813041
score_mean             -0.021964
Name: score, dtype: float64 [n=150] model=google/gemini-flash-1.5


### Save

In [33]:
df_llm.shape

(1, 18)

In [56]:
df_llm = pd.DataFrame(llm_info)
print(df_llm.shape, df3.shape)
# also join with df4
df_llm2 = (df_llm
          #  .drop(columns=['url'])
           .set_index('title2')
           .merge(df3, left_index=True, right_index=True)
              .sort_values('score', ascending=False)
            #   .drop(columns=['comments'])
            .reset_index(names='title2')
)
df_llm2


# now deduplicate by llm title and also keep only the ones that are fiction

(15032, 18) (15215, 8)


,title2,title,description,tags,reviews_quotes,reviews_summary,reccomendations,disrecommendations,why,if_you_liked_x_you_will_like_this,...,context,model,score,n_links,n_comments,comments,thread_urls,first_link_utc,last_link_utc,url
0,Worth the Candle,Worth the Candle,"Worth the Candle is a long, original, complete...","[web serial, LitRPG, isekai, portal fantasy, o...","[This is a self-insert litRPG portal fantasy, ...","Overall, Worth the Candle received overwhelmin...",Readers frequently praised the story's compell...,Some readers found the sheer length of the wor...,Readers of r/rational would likely enjoy Worth...,"[Mother of Learning, HPMOR, Worm, Erfworld, Lu...",...,\n\n----- Thread 10262 -----\n\n## [RT][WIP] W...,google/gemini-flash-1.5,1453,110,110,[{'author_flair_text': 'Self-Appointed Court S...,[https://reddit.com/r/rational/comments/7228po...,2017-07-28 13:17:36,2024-04-17 19:03:01,[https://archiveofourown.org/works/11478249/ch...
1,"Alexander Wales - The Metropolitan Man, Shadow...","Alexander Wales - The Metropolitan Man, Shadow...","Alexander Wales's works, including *Shadows of...","[web serial, rational fiction, fantasy, progre...",[Damn. That was like reading some of Sanderson...,"The reviews are overwhelmingly positive, with ...",Readers strongly recommend Alexander Wales's w...,Some readers express concerns about the infreq...,Readers of r/rational would likely enjoy Alexa...,"[Brandon Sanderson, Harry Potter, Worm, Mother...",...,\n\n----- Thread 10262 -----\n\n## [HF][DC] Sh...,google/gemini-flash-1.5,1408,22,22,[{'author_flair_text': 'Time flies like an arr...,[https://reddit.com/r/rational/comments/galt0r...,2015-04-18 18:13:47,2021-04-29 20:00:04,[https://www.patreon.com/alexanderwales]
2,Mother of Learning,Mother of Learning,Mother of Learning is a web serial about Zoria...,"[web serial, original fiction, fantasy, time l...","[With dread but cautious optimism, Enthusiasti...",Generally very positive; praised for clever pr...,"Readers praise its consistent internal logic, ...",Some readers find the writing style to be roug...,Readers of r/rational will appreciate the stor...,"[Time Braid, HPMOR, Worm]",...,\n\n----- Thread 10262 -----\n\n## [HF] Mother...,google/gemini-flash-1.5,1063,101,101,"[{'author_flair_text': 'Utopian Smut Peddler',...",[https://reddit.com/r/rational/comments/k8i7jg...,2014-07-26 08:21:12,2024-05-21 07:04:42,[https://www.fictionpress.com/s/2961893/1/Moth...
3,A Practical Guide to Evil,A Practical Guide to Evil,A Practical Guide to Evil is a long-running we...,"[web serial, fantasy, complete, grimdark, roma...",[It's probably one of the better written ones ...,"Generally very positive, praising the writing,...","Readers praise its well-developed characters, ...",Some readers find the frequent cliffhangers ir...,Readers of r/rational will likely enjoy this f...,"[Worm, Shadows of the Limelight]",...,\n\n----- Thread 10262 -----\n\n## [RT] A Prac...,google/gemini-flash-1.5,1025,92,92,[{'author_flair_text': 'Ankh-Morpork City Watc...,[https://reddit.com/r/rational/comments/ako900...,2016-04-05 18:53:21,2024-08-09 10:10:21,"[https://practicalguidetoevil.wordpress.com/, ..."
4,Worm,Worm,"Worm is a long, complete web serial (1.7 milli...","[web serial, complete, superhero, supervillain...",[I wouldn't really describe Worm as a rational...,"Worm receives overwhelmingly positive reviews,...",Readers recommend Worm for its compelling char...,"Some find the story too grim, dark, and depres...",Readers of r/rational might enjoy Worm for its...,"[HPMOR, A Practical Guide to Evil, Other Wildb...",...,\n\n----- Thread 10262 -----\n\n## Worm\n\n* A...,google/gemini-flash-1.5,925,83,83,[{'author_flair_text': 'Emergency Mustelid Hol...,[https://reddit.com/r/rational/comments/k8i7jg...,2014-01-27 02:09:42,2024-11-22 21:39:11,"[https://parahumans.wordpress.com/, https://pa..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15027,The Incredibles,The Incredib

In [ ]:
df_llm2

In [39]:
df_llm2_num = df_llm2.select_dtypes(include=['float64', 'int64'])
df_llm2_num['score_mean'] = df_llm2_num['score'] / df_llm2_num['n_links']
c = df_llm2_num.corr()['score']
print(f"Correlation with score ↑:\n{c}")

c = df_llm2_num.corr()['score_mean']
print(f"Correlation with score_mean ↑:\n{c}")

Correlation with score ↑:
rating_quality          0.100615
rating_rationality      0.058013
rating_writing          0.118364
rating_plot             0.129025
rating_character        0.125220
rating_worldbuilding    0.137975
score                   1.000000
n_links                 0.872121
n_comments              0.872121
score_mean              0.321027
Name: score, dtype: float64
Correlation with score_mean ↑:
rating_quality          0.102216
rating_rationality      0.056305
rating_writing          0.139368
rating_plot             0.141172
rating_character        0.135693
rating_worldbuilding    0.136899
score                   0.321027
n_links                 0.040730
n_comments              0.040730
score_mean              1.000000
Name: score_mean, dtype: float64


In [40]:
def format_flair(author_flair_text):
    if author_flair_text:
        return f" <em>{author_flair_text}</em>"
    return ""

import markdown
def commentmd2html(x: dict) -> str:
    body = markdown.markdown(x['body'])
    ts = pd.to_datetime(x['created_utc'], unit='s').strftime('%Y-%m-%d')
    flair = format_flair(x['author_flair_text'])
    url = prefix + x['permalink']
    s = f"""<h3><a href="{url}">{x.get('author', 'anon')} [{x['score']:+}] {flair} <sup>{ts}</sup></a></h3>
{body}
"""
    # print(s)
    return s

def collapsibe(title, body):
    return f"""<details><summary>{title}</summary>
{body}
</details>
"""

prefix = "https://reddit.com"
def c2md(x):
    return collapsibe(x['id'], commentmd2html(x))

# # QC test
# x = df3.iloc[0].comments[0]
# from IPython.display import display, HTML
# display(HTML(c2md(x)))

In [41]:
# First transform the fields for display
d = df3.reset_index().sort_values("score", ascending=False)


def url2a(url):
    text = url
    if "reddit.com/r/rational" in url:
        text = url.split("/")[-2]
        # text = url.replace('https://reddit.com/r/rational/comments/', '')

    return f'<a href="{url}">{text}</a>'


from collections import OrderedDict
def unique_elements(lst):
    return list(OrderedDict.fromkeys(lst))

def urls2a(urls, sep=None):
    if isinstance(urls, str):
        urls = urls.split("\n")

    # get uniques from list, keep in same order
    urls = unique_elements(urls)

    a_els = [url2a(u) for u in urls]

    # now make into a html list
    if sep is None:
        return "<ul>" + "".join([f"<li>{x}</li>" for x in a_els]) + "</ul>"
    else:
        return sep.join(a_els)


In [48]:

d = df_llm2.copy()

# make title have a link to first url
d['title'] = d.progress_apply(lambda x: f'<a href="{x["url"][0]}">{x["title"]}</a>', axis=1)
d['title2'] = d.progress_apply(lambda x: f'<a href="{x["url"][0]}">{x["title2"]}</a>', axis=1)

d["url"] = d["url"].apply(lambda x: collapsibe('...', urls2a(x)))
d["score"] = d["score"].round(2)

d['tags'] = d['tags'].apply(lambda x: ", ".join(x))
d['if_you_liked_x_you_will_like_this'] = d['if_you_liked_x_you_will_like_this'].apply(lambda x: ", ".join(x))

prefix = 'https://reddit.com/r/rational/comments/'
d["comment_urls"] = d['comments'].apply(lambda x: collapsibe("...", urls2a([prefix+y['permalink'] for y in x])))
d["thread_urls"] = d['thread_urls'].apply(lambda x: collapsibe("...", urls2a(x)))
# d['comments'] = d['comments'].progress_apply(lambda x: collapsibe("comments", "<br>".join([c2md(c) for c in x])))


d['first_link_utc'] = d['first_link_utc'].dt.strftime('%Y-%m-%d')
d['last_link_utc'] = d['last_link_utc'].dt.strftime('%Y-%m-%d')


  0%|          | 0/15032 [00:00<?, ?it/s]

  0%|          | 0/15032 [00:00<?, ?it/s]

In [49]:
print(d.columns)
rename_cols = OrderedDict(
    title2='Title (LLM)',
    title='Title',
    score='⬆️',
    n_comments='Comments',
    rating_quality='⭐Qual',
    rating_rationality='⭐Rat',
    rating_writing='⭐Writ',
    rating_plot='⭐Plot',
    rating_character='⭐Char',
    rating_worldbuilding='⭐World',
    tags='Tags',
    first_link_utc='First Link',
    last_link_utc='Last Link',
    n_links='Links',
    url='URLs',
    reviews_summary='Reviews Summary',
    thread_urls='Threads',
    comment_urls='Comments',
    if_you_liked_x_you_will_like_this='Similar',
    description='Description',
    reccomendations='Recommendations',
    disrecommendations='Disrecommendations',
    why='Why',
    reviews_quotes='Reviews',
)


dropping = [k for k in d.columns if k not in rename_cols.keys()]
print('Dropping', dropping)

d = d.drop(columns=['comments'])[list(rename_cols.keys())].rename(columns=rename_cols)

Index(['title2', 'title', 'description', 'tags', 'reviews_quotes',
       'reviews_summary', 'reccomendations', 'disrecommendations', 'why',
       'if_you_liked_x_you_will_like_this', 'rating_quality',
       'rating_rationality', 'rating_writing', 'rating_plot',
       'rating_character', 'rating_worldbuilding', 'context', 'model', 'score',
       'n_links', 'n_comments', 'comments', 'thread_urls', 'first_link_utc',
       'last_link_utc', 'url', 'comment_urls'],
      dtype='object')
Dropping ['context', 'model', 'comments']


In [50]:
hidden = ["comments", "first_link_utc", "last_link_utc",  'reviews_quotes', 'reviews_summary',
          'rating_writing', 'rating_plot', 'rating_character', 'rating_worldbuilding',

          'reccomendations', 'disrecommendations', 'title', 'n_links', 'n_comments'
]
# translate them using column rename
hidden2 = []
for h in hidden:
    if h in rename_cols:
        hidden2.append(rename_cols[h])
    hidden2.append(h)
# hidden = [k for k, v in rename_cols.items() if v in hidden]
hidden2

['comments',
 'First Link',
 'first_link_utc',
 'Last Link',
 'last_link_utc',
 'Reviews',
 'reviews_quotes',
 'Reviews Summary',
 'reviews_summary',
 '⭐Writ',
 'rating_writing',
 '⭐Plot',
 'rating_plot',
 '⭐Char',
 'rating_character',
 '⭐World',
 'rating_worldbuilding',
 'Recommendations',
 'reccomendations',
 'Disrecommendations',
 'disrecommendations',
 'Title',
 'title',
 'Links',
 'n_links',
 'Comments',
 'n_comments']

In [54]:
# put into the html template
import jinja2

environment = jinja2.Environment()
template = open("../index.jinja2.html").read()
template = environment.from_string(template)


data = d.to_json(orient="values")


columns = [
    {
        "title": c,
        "visible": c not in hidden2,
        "searchable": c not in hidden2,
        "footer": c,
        "name": c+"2",
    }
    for c in d.columns
]
columns = json.dumps(columns)


html = template.render(
    data=data,
    columns=columns,
)
html_out = Path("../index2.html").resolve()
open(html_out, "w").write(html)
columns

'[{"title": "Title (LLM)", "visible": true, "searchable": true, "footer": "Title (LLM)", "name": "Title (LLM)2"}, {"title": "Title", "visible": false, "searchable": false, "footer": "Title", "name": "Title2"}, {"title": "\\u2b06\\ufe0f", "visible": true, "searchable": true, "footer": "\\u2b06\\ufe0f", "name": "\\u2b06\\ufe0f2"}, {"title": "Comments", "visible": false, "searchable": false, "footer": "Comments", "name": "Comments2"}, {"title": "\\u2b50Qual", "visible": true, "searchable": true, "footer": "\\u2b50Qual", "name": "\\u2b50Qual2"}, {"title": "\\u2b50Rat", "visible": true, "searchable": true, "footer": "\\u2b50Rat", "name": "\\u2b50Rat2"}, {"title": "\\u2b50Writ", "visible": false, "searchable": false, "footer": "\\u2b50Writ", "name": "\\u2b50Writ2"}, {"title": "\\u2b50Plot", "visible": false, "searchable": false, "footer": "\\u2b50Plot", "name": "\\u2b50Plot2"}, {"title": "\\u2b50Char", "visible": false, "searchable": false, "footer": "\\u2b50Char", "name": "\\u2b50Char2"}, {

In [55]:
from IPython.display import HTML, display

htmla = f'<a href="{html_out}">View the page {html_out}</a>'
display(HTML(htmla))

In [53]:
df3.to_parquet("../outputs/df_links5.parquet")